Statistical Modeling & Risk-Based Pricing

Objective: Build and compare three models that predict claim severity
(TotalClaims for policies with a claim). The best model feeds into
a risk-based premium pricing formula.

Models: Linear Regression · Random Forest · XGBoost
Target: TotalClaims (claim severity — subset where TotalClaims > 0)
Metrics: RMSE (lower = better) and R² (higher = better)

── 0. Setup ──────────────────────────────────────────────────────────────────

In [1]:
import sys, pathlib
PROJECT_ROOT = pathlib.Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_data, engineer_base_features
from src.modeling import (
    prepare_features,
    train_linear,
    train_random_forest,
    train_xgboost,
    evaluate_model,
    compare_models,
    plot_shap_summary,
    interpret_shap,
)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
pd.set_option("display.float_format", "{:,.4f}".format)

── 1. Load & Prepare Data ────────────────────────────────────────────────────

In [2]:
DATA_PATH = "../data/insurance_data.txt"

df_raw = load_data(DATA_PATH)
df     = engineer_base_features(df_raw)

print(f"Full dataset  : {df.shape[0]:,} rows × {df.shape[1]} columns")
claimants = df[df["TotalClaims"] > 0]
print(f"Claimants only: {len(claimants):,} rows (policies with TotalClaims > 0)")

Full dataset  : 1,000,098 rows × 54 columns
Claimants only: 2,788 rows (policies with TotalClaims > 0)


── 2. Feature Engineering & Train/Test Split ─────────────────────────────────
prepare_features handles:
- subsetting to claimants (claims_only=True)
- engineering VehicleAge, HasAlarm, HasTracking
- dropping leakage columns (LossRatio, Margin, IDs)
- label-encoding categoricals
- imputing numeric NaNs with column median
- 80/20 train/test split

In [3]:
X_train, X_test, y_train, y_test = prepare_features(
    df,
    target="TotalClaims",
    claims_only=True,
    test_size=0.2,
)

print(f"\nTrain set: {X_train.shape[0]:,} rows, {X_train.shape[1]} features")
print(f"Test  set: {X_test.shape[0]:,} rows")
print(f"Target range: ZAR {y_train.min():,.0f} – {y_train.max():,.0f}")
print(f"Target mean : ZAR {y_train.mean():,.0f}")

# Visualise the target distribution (log scale because claims are right-skewed)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(y_train, bins=80, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].set_title("TotalClaims Distribution (raw)", fontweight="bold")
axes[0].set_xlabel("TotalClaims (ZAR)")
axes[0].set_ylabel("Count")

axes[1].hist(np.log1p(y_train), bins=80, color="darkorange", edgecolor="white", alpha=0.8)
axes[1].set_title("TotalClaims Distribution (log scale)", fontweight="bold")
axes[1].set_xlabel("log(1 + TotalClaims)")
axes[1].set_ylabel("Count")
plt.suptitle("Target Variable — Claim Severity", fontweight="bold")
plt.tight_layout()
plt.show()


Train set: 2,230 rows, 48 features
Test  set: 558 rows
Target range: ZAR 139 – 393,092
Target mean : ZAR 23,276


── 3. Model Training ─────────────────────────────────────────────────────────

── 3a. Linear Regression ─────────────────────────────────────────────────────
Baseline model. Assumes a linear relationship between features and
TotalClaims. Fast to train but typically under-performs on insurance data
due to non-linearities and heavy-tailed claim distributions.

In [4]:
print("\nTraining Linear Regression...")
lr_model = train_linear(X_train, y_train)
lr_result = evaluate_model(lr_model, X_test, y_test, model_name="Linear Regression")
print(f"  RMSE: {lr_result['rmse']:,.2f}  |  R²: {lr_result['r2']:.4f}")


Training Linear Regression...
  RMSE: 36,913.93  |  R²: 0.1527


── 3b. Random Forest ─────────────────────────────────────────────────────────
Ensemble of 200 decision trees. Handles non-linearities and feature
interactions naturally. More robust to outliers than Linear Regression.
min_samples_leaf=10 prevents overfitting on noisy claim tails.

In [5]:
print("Training Random Forest...")
rf_model = train_random_forest(X_train, y_train, n_estimators=200, max_depth=15)
rf_result = evaluate_model(rf_model, X_test, y_test, model_name="Random Forest")
print(f"  RMSE: {rf_result['rmse']:,.2f}  |  R²: {rf_result['r2']:.4f}")

Training Random Forest...
  RMSE: 33,854.87  |  R²: 0.2873


── 3c. XGBoost ───────────────────────────────────────────────────────────────
Gradient-boosted trees. Builds trees sequentially, each correcting the
errors of the previous. Typically achieves the best accuracy on tabular
insurance data. learning_rate=0.05 with 400 estimators balances bias/variance.

In [6]:
print("Training XGBoost...")
xgb_model = train_xgboost(X_train, y_train)
xgb_result = evaluate_model(xgb_model, X_test, y_test, model_name="XGBoost")
print(f"  RMSE: {xgb_result['rmse']:,.2f}  |  R²: {xgb_result['r2']:.4f}")

Training XGBoost...
  RMSE: 37,582.13  |  R²: 0.1218


── 4. Model Comparison Table ─────────────────────────────────────────────────

In [7]:
print("\n" + "="*50)
print("MODEL COMPARISON (sorted by RMSE)")
print("="*50)

comparison = compare_models([lr_result, rf_result, xgb_result])
print(comparison.to_string())

# Visualise RMSE and R² side by side
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
models = comparison["model"].tolist()
rmses  = comparison["rmse"].tolist()
r2s    = comparison["r2"].tolist()

bar_colors = ["#d62728", "#ff7f0e", "#2ca02c"]  # red=worst, green=best by RMSE order
axes[0].barh(models, rmses, color=bar_colors[::-1], edgecolor="white")
axes[0].set_title("RMSE (lower = better)", fontweight="bold")
axes[0].set_xlabel("RMSE (ZAR)")
for i, v in enumerate(rmses):
    axes[0].text(v * 0.98, i, f"{v:,.0f}", va="center", ha="right",
                 color="white", fontweight="bold", fontsize=9)

axes[1].barh(models, r2s, color=bar_colors[::-1], edgecolor="white")
axes[1].set_title("R² (higher = better)", fontweight="bold")
axes[1].set_xlabel("R²")
axes[1].set_xlim(min(0, min(r2s)) - 0.05, 1.0)
for i, v in enumerate(r2s):
    axes[1].text(max(v - 0.02, 0.01), i, f"{v:.4f}", va="center", ha="right",
                 color="white", fontweight="bold", fontsize=9)

plt.suptitle("Model Performance Comparison — Claim Severity", fontweight="bold")
plt.tight_layout()
plt.show()


MODEL COMPARISON (sorted by RMSE)
               model        rmse     r2  n_test
1      Random Forest 33,854.8700 0.2873     558
2  Linear Regression 36,913.9300 0.1527     558
3            XGBoost 37,582.1300 0.1218     558


── 5. Residual Analysis (Best Model) ─────────────────────────────────────────
Residuals = actual − predicted. A good model has residuals centred at 0
with no clear pattern. If residuals fan out at higher predicted values,
the model under-estimates large claims (heteroscedasticity).

In [8]:
best_model      = xgb_model    # update if RF wins
best_model_name = "XGBoost"

y_pred = best_model.predict(X_test)
residuals = y_test.values - y_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(y_pred, residuals, alpha=0.3, s=8, color="steelblue")
axes[0].axhline(0, color="red", linestyle="--", lw=1.2)
axes[0].set_title(f"{best_model_name} — Residuals vs Predicted", fontweight="bold")
axes[0].set_xlabel("Predicted TotalClaims (ZAR)")
axes[0].set_ylabel("Residual (Actual − Predicted)")

axes[1].hist(residuals, bins=80, color="coral", edgecolor="white", alpha=0.8)
axes[1].axvline(0, color="black", lw=1.2)
axes[1].set_title("Residual Distribution", fontweight="bold")
axes[1].set_xlabel("Residual (ZAR)")
axes[1].set_ylabel("Count")
plt.suptitle(f"Residual Analysis — {best_model_name}", fontweight="bold")
plt.tight_layout()
plt.show()

# Actual vs Predicted scatter (clipped to 99th percentile for readability)
clip_val = np.percentile(y_test, 99)
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(
    y_test.clip(upper=clip_val),
    np.clip(y_pred, 0, clip_val),
    alpha=0.2, s=8, color="steelblue"
)
ax.plot([0, clip_val], [0, clip_val], "r--", lw=1.2, label="Perfect prediction")
ax.set_title(f"Actual vs Predicted — {best_model_name} (clipped at 99th pct)",
             fontweight="bold")
ax.set_xlabel("Actual TotalClaims (ZAR)")
ax.set_ylabel("Predicted TotalClaims (ZAR)")
ax.legend()
plt.tight_layout()
plt.show()

── 6. Feature Importance — SHAP ──────────────────────────────────────────────
SHAP (SHapley Additive exPlanations) assigns each feature a contribution
to each individual prediction. This is more trustworthy than built-in
feature_importances_ because it accounts for feature interactions.

The beeswarm plot shows:
Y-axis: features ranked by total impact (most important at top).
X-axis: SHAP value — positive = pushes prediction higher,
negative = pushes prediction lower.
Colour: feature value (red = high, blue = low).

In [9]:
print("\nComputing SHAP values (this may take ~30 seconds)...")

# Use a sample of 2000 test rows for SHAP speed on large datasets
shap_sample = X_test.sample(min(2000, len(X_test)), random_state=42)

fig_shap = plot_shap_summary(best_model, shap_sample, model_name=best_model_name)
plt.show()


Computing SHAP values (this may take ~30 seconds)...


── 7. Top Feature Interpretations ────────────────────────────────────────────

In [10]:
print("\nTop 10 Features by Mean |SHAP| Value:")
shap_importance = interpret_shap(best_model, shap_sample, top_n=10)
print(shap_importance.to_string())

# Bar chart of SHAP importance
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(
    shap_importance["feature"][::-1],
    shap_importance["mean_abs_shap"][::-1],
    color="steelblue", edgecolor="white"
)
ax.set_title(f"Top 10 Features — Mean |SHAP| Value\n{best_model_name}",
             fontweight="bold")
ax.set_xlabel("Mean |SHAP| (ZAR impact on prediction)")
plt.tight_layout()
plt.show()


Top 10 Features by Mean |SHAP| Value:
                     feature  mean_abs_shap
1                 SumInsured     9,943.5342
2   CalculatedPremiumPerTerm     7,324.2090
3               TotalPremium     4,562.6733
4                 PostalCode     1,935.0793
5                     mmcode     1,718.5441
6        CustomValueEstimate     1,433.1990
7         CapitalOutstanding     1,089.3239
8              cubiccapacity       952.5248
9           RegistrationYear       876.8930
10             NumberOfDoors       834.2816


── 8. Risk-Based Pricing Framework ───────────────────────────────────────────
A simple actuarial pricing formula combining:
- P(claim)         : claim frequency (proportion with claims)
- Predicted Severity: from the best model
- Expense loading  : 15 % of pure premium (industry typical)
- Profit margin    : 10 %

Premium = (P(claim) × Predicted_Severity × (1 + expense_loading)) / (1 - profit_margin)

In [11]:
claim_freq    = (df["TotalClaims"] > 0).mean()
expense_load  = 0.15
profit_margin = 0.10

# Predict severity for all claimant policies in the test set
pure_premium = claim_freq * y_pred
recommended_premium = (pure_premium * (1 + expense_load)) / (1 - profit_margin)

print("\n" + "="*50)
print("RISK-BASED PRICING FRAMEWORK")
print("="*50)
print(f"Portfolio claim frequency : {claim_freq:.2%}")
print(f"Expense loading           : {expense_load:.0%}")
print(f"Profit margin             : {profit_margin:.0%}")
print(f"\nTest-set recommended premiums (ZAR):")
print(f"  Mean   : {recommended_premium.mean():,.2f}")
print(f"  Median : {np.median(recommended_premium):,.2f}")
print(f"  Min    : {recommended_premium.min():,.2f}")
print(f"  Max    : {recommended_premium.max():,.2f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(
    np.clip(recommended_premium, 0, np.percentile(recommended_premium, 99)),
    bins=80, color="darkorange", edgecolor="white", alpha=0.85
)
ax.axvline(recommended_premium.mean(), color="red", linestyle="--", lw=1.5,
           label=f"Mean: ZAR {recommended_premium.mean():,.0f}")
ax.set_title("Distribution of Model-Recommended Premiums (Test Set)",
             fontweight="bold")
ax.set_xlabel("Recommended Premium (ZAR)")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()

print("\nTask 4 complete.")


RISK-BASED PRICING FRAMEWORK
Portfolio claim frequency : 0.28%
Expense loading           : 15%
Profit margin             : 10%

Test-set recommended premiums (ZAR):
  Mean   : 78.83
  Median : 25.37
  Min    : -0.65
  Max    : 698.29

Task 4 complete.
